In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
NUM_CLASSES = 3

model = models.resnet50(weights=None)  # no need to download weights again
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
model = model.to(device)

In [4]:
try:
    model.load_state_dict(torch.load("best_model_finetuned.pth", map_location=device))
    print("Loaded fine-tuned model (best_model_finetuned.pth)")
except FileNotFoundError:
    try:
        model.load_state_dict(torch.load("best_model.pth", map_location=device))
        print("Loaded base model (best_model.pth)")
    except FileNotFoundError:
        raise FileNotFoundError("No trained model found. Please run training first to generate best_model.pth or best_model_finetuned.pth")

model.eval()

Loaded fine-tuned model (best_model_finetuned.pth)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [5]:
IMG_SIZE = 224

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean
        std=[0.229, 0.224, 0.225]     # ImageNet std
    )
])

In [6]:
class_names = ['normal', 'osteopenia', 'osteoporosis']

In [7]:
def predict_image(img_path):
    """Predict class for a single image."""
    image = Image.open(img_path).convert("RGB")
    image = val_transforms(image).unsqueeze(0).to(device)   # add batch dimension

    with torch.no_grad():
        output = model(image)
        _, pred = torch.max(output, 1)

    return class_names[pred.item()]

In [11]:
result = predict_image("Osteoporosis 122.jpg")
print(f"Prediction: {result}")

Prediction: osteoporosis
